In [1]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader

# 导入你写好的模块
from models import CLIPModel
from modules import AdaptiveRegionGenerator, visualize_adaptive_regions
from dataset import CLIPDataset

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("正在加载切片 3 的数据集...")
dataset_test = CLIPDataset(
    image_path="/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_C1_Merged.tif",
    spatial_pos_path="/root/disk2/runzhi/BLEEP/GSE240429_data/data/tissue_pos_matrices/tissue_positions_list_3.csv",
    reduced_mtx_path="/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/harmony_matrix.npy",
    barcode_path="/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices/3/barcodes.tsv",
    is_train=False
)
# batch_size 可以设大点跑得快
test_loader = DataLoader(dataset_test, batch_size=64, shuffle=False, num_workers=4)
print(f"数据加载完成，共 {len(dataset_test)} 个斑点/图块。")

/root/disk2/runzhi/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/disk2/runzhi/.conda/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


正在加载切片 3 的数据集...


ValueError: Barcodes / reduced matrix cell-axis mismatch. reduced_matrix cells=0 vs filtered barcodes=2277.

In [ ]:
# ==========================================
# 💡 调试区：在这里修改你的 ARG 参数！
# ==========================================
GRID_SIZE = 16   # 尝试调大，比如 16, 24
TOP_K = 2        # 尝试调小，比如 2 或 1 (强制空间约束)
FEATURE_DIM = 2048 # ResNet50 的输出维度

# 1. 初始化模型
# 1. 初始化模型并加载【真实训练权重】！
model = CLIPModel().to(device)

# 替换成你实际保存 best.pt 的路径
checkpoint_path = "/root/disk2/runzhi/BLEEP/bleep_paper_params/best.pt" 

# 加载 DDP 保存的权重（需要去掉 'module.' 前缀）
state_dict = torch.load(checkpoint_path, map_location=device)
new_state_dict = {}
for key in state_dict.keys():
    new_key = key.replace('module.', '') 
    new_state_dict[new_key] = state_dict[key]

# 把权重喂给模型，并开启 eval 模式
model.load_state_dict(new_state_dict, strict=False)
model.eval()

# 2. 初始化 ARG
arg_generator = AdaptiveRegionGenerator(
    feature_dim=FEATURE_DIM,
    grid_size=GRID_SIZE,
    topk=TOP_K
).to(device)
arg_generator.eval()

# 3. 提取特征和坐标
all_image_features = []
all_coords = []

print("正在提取图像特征和坐标...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        img = batch["image"].to(device)
        
        # 🚨 修复报错：处理 Dataloader 返回 list 的情况
        raw_coords = batch["spatial_coords"]
        if isinstance(raw_coords, list) or isinstance(raw_coords, tuple):
            # 将 [tensor(x_batch), tensor(y_batch)] 拼成 (B, 2) 形状的张量
            coords = torch.stack(raw_coords, dim=1).to(device).float()
        else:
            coords = raw_coords.to(device).float()
        
        # 确保坐标形状是对的 (B, 2)
        if coords.shape[1] != 2 and coords.shape[0] == 2:
            coords = coords.t()
            
        # 过 ResNet
        features = model.image_encoder(img)
        
        all_image_features.append(features)
        all_coords.append(coords)

all_image_features = torch.cat(all_image_features, dim=0)
all_coords = torch.cat(all_coords, dim=0)

# 4. 送入 ARG 获取区域划分结果！
print("正在执行自适应区域划分...")
with torch.no_grad():
    z_he_valid, region_indices = arg_generator(all_image_features, all_coords)

region_indices_np = region_indices.cpu().numpy()
patch_coords_np = all_coords.cpu().numpy()

print(f"划分完毕！共生成了 {z_he_valid.shape[0]} 个有效不规则微环境区域。")

In [ ]:
# 1. 读取原切片缩略图作为底图
wsi_path = "/root/disk2/runzhi/BLEEP/GSE240429_data/images/GEX_C73_C1_Merged.tif"
wsi_thumbnail = cv2.imread(wsi_path)
wsi_thumbnail = cv2.cvtColor(wsi_thumbnail, cv2.COLOR_BGR2RGB)

# 如果原图太大，强行缩放一下（比如缩放到长边 2000 像素左右）防止内存爆炸
H, W = wsi_thumbnail.shape[:2]
scale_factor = 2000.0 / max(H, W)
if scale_factor < 1.0:
    new_size = (int(W * scale_factor), int(H * scale_factor))
    wsi_thumbnail = cv2.resize(wsi_thumbnail, new_size)
    # 坐标也要跟着缩放！
    patch_coords_np = patch_coords_np * scale_factor

# 2. 调用可视化核心函数
# ==========================================
# 💡 绘图调试区：调节 patch_size 和 sigma
# ==========================================
PATCH_SIZE = 32   # 尝试调大，直到图块之间没有明显的缝隙
SIGMA = 4.0       # 高斯平滑系数，越大边缘越水润连贯
ALPHA = 0.55      # 颜色遮罩透明度

print("正在渲染可视化图像...")
# 🚨 修复报错：只接收一个返回值
overlay_img = visualize_adaptive_regions(
    patch_coords=patch_coords_np,
    region_indices=region_indices_np,
    wsi_thumbnail=wsi_thumbnail,
    patch_size=PATCH_SIZE,  
    sigma=SIGMA,
    alpha=ALPHA
)

# 3. 在 Notebook 里展示结果
plt.figure(figsize=(15, 15), dpi=150)
plt.imshow(overlay_img)
plt.axis('off')
plt.title(f"ARG Visualization (Grid={GRID_SIZE}, TopK={TOP_K})")
plt.show()

# 顺手保存一张到本地
cv2.imwrite("/root/disk2/runzhi/BLEEP/result/debug_arg_vis.png", cv2.cvtColor(overlay_img, cv2.COLOR_RGB2BGR))

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
import numpy as np

# 1. 从 test_loader 中抽出一个 Batch 的数据
batch = next(iter(test_loader))
images = batch["image"]          # 形状通常是 (B, 3, 224, 224)
raw_coords = batch["spatial_coords"]

# 处理坐标格式
if isinstance(raw_coords, (list, tuple)):
    coords = torch.stack(raw_coords, dim=1).cpu().numpy()
else:
    coords = raw_coords.cpu().numpy()

# 2. 逆归一化 (Un-normalize)
# 喂给 ResNet 的图像通常经过了 ImageNet 的标准化处理，直接画会发黑发蓝
# 我们需要把它还原成肉眼可见的 RGB 图像
inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# 3. 画一个 4x4 的网格，查看前 16 个图块
fig, axes = plt.subplots(4, 4, figsize=(12, 12), dpi=100)
fig.suptitle("Actual Patches Fed to Image Encoder (ResNet50)", fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < len(images):
        # 取出单张图片并反归一化
        img_tensor = inv_normalize(images[i])
        
        # 将 PyTorch 的 (C, H, W) 转换成 matplotlib 需要的 (H, W, C)
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        
        # 截断超出 [0, 1] 范围的像素（防止高光溢出报错）
        img_np = np.clip(img_np, 0, 1)
        
        ax.imshow(img_np)
        
        # 把该图块在原 WSI 上的物理坐标打在标题上
        cx, cy = coords[i]
        ax.set_title(f"X: {cx:.0f}, Y: {cy:.0f}", fontsize=10)
        
    ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()